# Libraries import

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Data load

In [ ]:
df = pd.read_csv('data/accident_prediction_india.csv')
df.shape

# Inspect

In [ ]:
df.head()

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
df.isnull().sum()

In [ ]:
df.duplicated().sum()

# Null % Analysis

In [ ]:
null_counts = df.isnull().sum()
null_percentages = (df.isnull().sum() / df.shape[0]) * 100

null_summary = pd.DataFrame({
    'Null Count': null_counts,
    'Null Percentage': null_percentages.round(2)
})
null_summary = null_summary[null_summary['Null Count'] > 0].sort_values('Null Percentage', ascending=False)
print(null_summary)

high_null_cols = null_summary[null_summary['Null Percentage'] > 20]
print("\nColumns exceeding 20% null rate:")
print(high_null_cols if not high_null_cols.empty else "None")

# Nulls handling

In [ ]:
df['Traffic Control Presence'] = df['Traffic Control Presence'].fillna('None')
df['Driver License Status'] = df['Driver License Status'].fillna('Unknown')

# Median check for numeric columns

In [ ]:
numeric_cols = df.select_dtypes(include=['int64', 'float64']).columns
print("Nulls in numeric columns:")
for col in numeric_cols:
    print(f"{col}: {df[col].isnull().sum()} nulls")

# Confirm nulls

In [ ]:
df.isnull().sum().sum()

# Duplicates remove

In [ ]:
rows_before = df.shape[0]
duplicate_count = df.duplicated().sum()
print(f"Duplicate rows found: {duplicate_count}")

df = df.drop_duplicates()
rows_after = df.shape[0]
print(f"Rows removed: {rows_before - rows_after}")

print("\nNull % after duplicate removal:")
print((df.isnull().sum() / df.shape[0] * 100).round(2))

# Verify Data types 

In [ ]:
df.dtypes

# Data Type Correction
#### Checking all columns for dtype correctness. No column in this dataset has a genuinely incorrect dtype (all numeric fields are stored as int64/float64, all categorical fields as object). To demonstrate the conversion technique, we show how `pd.to_numeric()` with `errors='coerce'` would handle a miscast column, and additionally convert repetitive string columns to the memory-efficient `category` dtype.

In [ ]:
memory_before = df.memory_usage(deep=True).sum()
print(f"Memory usage BEFORE category conversion: {memory_before / 1024:.2f} KB")

In [ ]:
category_cols = ['State Name', 'Weather Conditions', 'Road Type', 'Road Condition', 
                  'Lighting Conditions', 'Traffic Control Presence', 'Driver Gender', 
                  'Driver License Status', 'Alcohol Involvement', 'Vehicle Type Involved',
                  'Accident Severity', 'Day of Week']

for col in category_cols:
    df[col] = df[col].astype('category')

df.dtypes

In [ ]:
memory_after = df.memory_usage(deep=True).sum()
print(f"Memory usage AFTER category conversion: {memory_after / 1024:.2f} KB")
print(f"Memory reduced by: {(memory_before - memory_after) / 1024:.2f} KB ({(1 - memory_after/memory_before)*100:.1f}%)")

In [ ]:
# Demonstration: showing the technique using a temporary miscast copy
demo_col = df['Speed Limit (km/h)'].astype(str)  
print(f"Before: dtype = {demo_col.dtype}")

demo_col_fixed = pd.to_numeric(demo_col, errors='coerce')
print(f"After pd.to_numeric(): dtype = {demo_col_fixed.dtype}")
print(f"Any values lost/coerced to NaN: {demo_col_fixed.isnull().sum()}")

# Descriptive Statistics & Skewness
#### Computing summary statistics and skewness for all numeric columns to understand the shape of each distribution.

In [ ]:
numeric_cols = df.select_dtypes(include=['int64', 'float64']).columns
print(df[numeric_cols].describe())

print("\n--- Skewness ---")
skewness = df[numeric_cols].skew().sort_values(ascending=False)
print(skewness)

most_skewed_col = skewness.abs().idxmax()
print(f"\nColumn with highest absolute skewness: {most_skewed_col} (skew = {skewness[most_skewed_col]:.3f})")

# Outlier Detection using IQR
#### Detecting outliers in two numeric columns using the Interquartile Range (IQR) method.

In [ ]:
def detect_outliers_iqr(df, col):
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    outliers = df[(df[col] < lower_bound) | (df[col] > upper_bound)]
    print(f"\n--- {col} ---")
    print(f"Q1: {Q1}, Q3: {Q3}, IQR: {IQR}")
    print(f"Lower bound: {lower_bound}, Upper bound: {upper_bound}")
    print(f"Number of outliers: {len(outliers)}")
    return outliers

outliers_speed = detect_outliers_iqr(df, 'Speed Limit (km/h)')
outliers_age = detect_outliers_iqr(df, 'Driver Age')

# EDA

## Plot 1 — Severity Distribution

In [ ]:
sns.set_theme(style="whitegrid", palette="viridis")
plt.rcParams['figure.figsize'] = (8, 5)
plt.rcParams['font.size'] = 11

plt.figure(figsize=(8, 5))
ax = sns.countplot(
    data=df, 
    x='Accident Severity', 
    hue='Accident Severity',
    order=df['Accident Severity'].value_counts().index,
    palette='rocket',       # try 'mako', 'flare', 'crest' also
    legend=False,
    edgecolor='black',
    linewidth=1
)
for container in ax.containers:
    ax.bar_label(container, fontsize=11, fontweight='bold', padding=3)

plt.title('Accident Severity Distribution', fontsize=15, fontweight='bold', pad=15)
plt.xlabel('Accident Severity', fontsize=12)
plt.ylabel('Number of Accidents', fontsize=12)
sns.despine(left=True, bottom=True)   # removes chart border lines, cleaner look
plt.tight_layout()
plt.savefig('plots/severity_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

## Plot 2 — Alcohol Involvement vs Severity

In [ ]:
plt.figure(figsize=(8, 5))
ax = sns.countplot(
    data=df, 
    x='Alcohol Involvement', 
    hue='Accident Severity',
    palette='rocket',
    edgecolor='black',
    linewidth=0.8
)

for container in ax.containers:
    ax.bar_label(container, fontsize=9, padding=2)

plt.title('Accident Severity by Alcohol Involvement', fontsize=15, fontweight='bold', pad=15)
plt.xlabel('Alcohol Involved', fontsize=12)
plt.ylabel('Number of Accidents', fontsize=12)
plt.legend(title='Severity')
sns.despine(left=True, bottom=True)
plt.tight_layout()
plt.savefig('plots/severity_by_alcohol.png', dpi=150, bbox_inches='tight')
plt.show()

## Plot 3 — Lighting Conditions vs Severity

In [ ]:
plt.figure(figsize=(9, 5))
ax = sns.countplot(
    data=df, 
    x='Lighting Conditions', 
    hue='Accident Severity',
    palette='rocket',
    edgecolor='black',
    linewidth=0.8
)

for container in ax.containers:
    ax.bar_label(container, fontsize=8, padding=2)

plt.title('Accident Severity by Lighting Conditions', fontsize=15, fontweight='bold', pad=15)
plt.xlabel('Lighting Conditions', fontsize=12)
plt.ylabel('Number of Accidents', fontsize=12)
plt.xticks(rotation=20)
plt.legend(title='Severity')
sns.despine(left=True, bottom=True)
plt.tight_layout()
plt.savefig('plots/severity_by_lighting.png', dpi=150, bbox_inches='tight')
plt.show()

## Plot 4 — Number of Casualties Distribution

In [ ]:
plt.figure(figsize=(8, 5))
sns.histplot(df['Driver Age'], bins=20, kde=True, color='crimson', edgecolor='black')
plt.title('Distribution of Driver Age (Most Skewed Column)', fontsize=15, fontweight='bold', pad=15)
plt.xlabel('Driver Age', fontsize=12)
plt.ylabel('Frequency', fontsize=12)
sns.despine(left=True, bottom=True)
plt.tight_layout()
plt.savefig('plots/histogram_driver_age.png', dpi=150, bbox_inches='tight')
plt.show()

## Plot 5 — Correlation Heatmap

In [ ]:
plt.figure(figsize=(8, 6))
numeric_cols = ['Number of Vehicles Involved', 'Number of Casualties', 
                 'Number of Fatalities', 'Speed Limit (km/h)', 'Driver Age']

corr = df[numeric_cols].corr()
sns.heatmap(corr, annot=True, cmap='rocket_r', fmt='.2f', linewidths=0.5, 
            square=True, cbar_kws={"shrink": 0.8})

plt.title('Correlation Heatmap — Numeric Features', fontsize=15, fontweight='bold', pad=15)
plt.tight_layout()
plt.savefig('plots/correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

## Plot 6 — Top 10 States by Accident Count

In [ ]:
plt.figure(figsize=(9, 6))
top_states = df['State Name'].value_counts().head(10)

top_states.index = top_states.index.astype(str)

ax = sns.barplot(
    x=top_states.values, 
    y=top_states.index, 
    hue=top_states.index,      
    palette='rocket', 
    legend=False,               
    edgecolor='black'
)

for i, v in enumerate(top_states.values):
    ax.text(v + 5, i, str(v), va='center', fontsize=10, fontweight='bold')

plt.title('Top 10 States by Accident Count', fontsize=15, fontweight='bold', pad=15)
plt.xlabel('Number of Accidents', fontsize=12)
plt.ylabel('State', fontsize=12)
sns.despine(left=True, bottom=True)
plt.tight_layout()
plt.savefig('plots/top_states.png', dpi=150, bbox_inches='tight')
plt.show()

## Plot 7 — Casualty Trend Across Accident Records

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(df.index[:200], df['Number of Casualties'][:200], color='crimson', linewidth=1)
plt.title('Number of Casualties Across First 200 Accident Records', fontsize=14, fontweight='bold')
plt.xlabel('Row Index')
plt.ylabel('Number of Casualties')
plt.tight_layout()
plt.savefig('plots/line_plot_casualties.png', dpi=150, bbox_inches='tight')
plt.show()

## Plot 8 — Speed Limit vs Number of Casualties

In [ ]:
plt.figure(figsize=(8, 6))
sns.scatterplot(data=df, x='Speed Limit (km/h)', y='Number of Casualties', alpha=0.4, color='darkred')
plt.title('Speed Limit vs Number of Casualties', fontsize=14, fontweight='bold')
plt.xlabel('Speed Limit (km/h)')
plt.ylabel('Number of Casualties')
plt.tight_layout()
plt.savefig('plots/scatter_speed_casualties.png', dpi=150, bbox_inches='tight')
plt.show()

## Plot 9 — Driver Age Spread Across Severity Categories

In [ ]:
plt.figure(figsize=(9, 6))
sns.boxplot(data=df, x='Accident Severity', y='Driver Age', hue='Accident Severity', palette='rocket', legend=False)
plt.title('Driver Age Distribution by Accident Severity', fontsize=14, fontweight='bold')
plt.xlabel('Accident Severity')
plt.ylabel('Driver Age')
plt.tight_layout()
plt.savefig('plots/boxplot_age_severity.png', dpi=150, bbox_inches='tight')
plt.show()

## Plot 10 — Average Casualties by Road Type

In [ ]:
plt.figure(figsize=(9, 5))
road_avg_casualties = df.groupby('Road Type', observed=True)['Number of Casualties'].mean().sort_values(ascending=False)
road_avg_casualties.index = road_avg_casualties.index.astype(str)

ax = sns.barplot(
    x=road_avg_casualties.values, 
    y=road_avg_casualties.index, 
    hue=road_avg_casualties.index,
    palette='rocket', 
    legend=False,
    edgecolor='black'
)

for i, v in enumerate(road_avg_casualties.values):
    ax.text(v + 0.05, i, f"{v:.2f}", va='center', fontsize=10, fontweight='bold')

plt.title('Average Casualties per Accident by Road Type', fontsize=15, fontweight='bold', pad=15)
plt.xlabel('Average Number of Casualties', fontsize=12)
plt.ylabel('Road Type', fontsize=12)
sns.despine(left=True, bottom=True)
plt.tight_layout()
plt.savefig('plots/bar_mean_casualties_by_roadtype.png', dpi=150, bbox_inches='tight')
plt.show()

# Mean vs Median Comparison

## Imputation Strategy Comparison — Mean vs Median
#### For the two highest-skewness columns, comparing mean and median before deciding an imputation strategy.

In [ ]:
top_skew_cols = ['Driver Age', 'Year']

for col in top_skew_cols:
    col_mean = df[col].mean()
    col_median = df[col].median()
    print(f"{col}: Mean = {col_mean:.2f}, Median = {col_median:.2f}, Skew = {df[col].skew():.4f}")

print("\nNulls remaining in these columns:")
print(df[top_skew_cols].isnull().sum())

## Spearman vs Pearson Correlation

In [ ]:
numeric_cols = df.select_dtypes(include=['int64', 'float64']).columns

pearson_corr = df[numeric_cols].corr(method='pearson')
spearman_corr = df[numeric_cols].corr(method='spearman')

diff = (spearman_corr - pearson_corr).abs()

diff_pairs = diff.where(np.triu(np.ones(diff.shape), k=1).astype(bool)).stack().sort_values(ascending=False)

print("Top 3 pairs with largest |Spearman - Pearson| difference:\n")
print(diff_pairs.head(3))

print("\n--- Pearson Correlation Matrix ---")
print(pearson_corr)

print("\n--- Spearman Correlation Matrix ---")
print(spearman_corr)

## Grouped Aggregation — Casualties by State

In [ ]:
agg_result = df.groupby('State Name', observed=True)['Number of Casualties'].agg(['mean', 'std', 'count'])
agg_result = agg_result.sort_values('mean', ascending=False)
print(agg_result)

highest_mean_state = agg_result['mean'].idxmax()
highest_std_state = agg_result['std'].idxmax()
mean_ratio = agg_result['mean'].max() / agg_result['mean'].min()

print(f"\nHighest mean: {highest_mean_state} ({agg_result['mean'].max():.2f})")
print(f"Highest std: {highest_std_state} ({agg_result['std'].max():.2f})")
print(f"Ratio of highest to lowest mean: {mean_ratio:.3f}")

## Save Cleaned Dataset

In [ ]:
df.to_csv('cleaned_data.csv', index=False)
print("Saved cleaned_data.csv")